In [1]:
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
import torch

import warnings
warnings.filterwarnings('ignore')

## Prepare data

In [2]:
# скачиваем данные - тут готовый датасет в нужном формате
dataset = load_dataset("blinoff/kinopoisk")
dataset = dataset.rename_column("grade3", "labels").class_encode_column("labels")

# по дефолту тут нет тестовых данных - получим их
split_data = dataset["train"].train_test_split(test_size=0.2, seed=42, shuffle=True)

README.md: 0.00B [00:00, ?B/s]

kinopoisk.jsonl:   0%|          | 0.00/143M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/36591 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/36591 [00:00<?, ? examples/s]

## Select model and load tokenizer

In [3]:
MODEL_NAME = "DeepPavlov/rubert-base-cased"
# Модель
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME,
                                                           num_labels=len(dataset["train"].features["labels"].names)
                                                           )
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [4]:
# Проверяем, доступна ли видеокарта, и перемещаем модель
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model.to(device)

cuda


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [5]:
# text = "Это пример текста."
# inputs = tokenizer(text, return_tensors='pt')
# print(inputs['input_ids'])
# print(inputs['attention_mask'])

# texts = ["Первый текст.", "Второй, но гораздо более длинный текст для примера."]

# inputs = tokenizer(
#     texts,
#     padding=True,            # Выравниваем длины в пределах батча
#     truncation=True,         # Обрезаем текст до max_length
#     max_length=512,          # Максимальная длина последовательности
#     return_tensors='pt'      # Получаем PyTorch тензоры
# )

## Tokenizer

In [6]:
def tokenize_batch(batch):
    return tokenizer(
        batch['content'],          # Название колонки с текстом в вашем датасете
        padding='max_length',   # Фиксируем длину для всех примеров (подготовка для батчей)
        truncation=True,
        max_length=256,
        return_tensors='pt'
    )

In [7]:
tokenized_train = split_data["train"].map(
    tokenize_batch,
    batched=True,            # Включаем пакетную обработку (быстрее)
    batch_size=1000          # Оптимальный размер батча для вашего датасета
)

tokenized_val = split_data["test"].map(
    tokenize_batch,
    batched=True,            # Включаем пакетную обработку (быстрее)
    batch_size=1000          # Оптимальный размер батча для вашего датасета
)

Map:   0%|          | 0/29272 [00:00<?, ? examples/s]

Map:   0%|          | 0/7319 [00:00<?, ? examples/s]

## Check class balance and resample for smoke test

In [8]:
from collections import Counter

# Собираем все метки из dataset
labels = tokenized_train['labels']
counter = Counter(labels)
print("Распределение классов:", dict(counter))
print("Всего примеров:", len(labels))

Распределение классов: {0: 3801, 1: 21805, 2: 3666}
Всего примеров: 29272


In [9]:
#специально понижаю размерность
small_train = tokenized_train.shuffle(seed=42)
small_train = tokenized_train.select(range(300))

small_val = tokenized_val.shuffle(seed=42)
small_val = tokenized_val.select(range(300))

In [ ]:
small_train.shape
medium_train = tokenized_train.select(range(10000))
medium_train.shape

In [10]:
labels = small_train['labels']
counter = Counter(labels)
print("Распределение классов:", dict(counter))
print("Всего примеров:", len(labels))

Распределение классов: {0: 45, 1: 227, 2: 28}
Всего примеров: 300


In [11]:
def downsample_dataset(dataset, label_col='labels'):
    """Уменьшает каждый класс до минимального количества примеров среди классов."""
    label_indices = {}
    for idx, label in enumerate(dataset[label_col]):
        label_indices.setdefault(label, []).append(idx)

    min_count = min(len(indices) for indices in label_indices.values())
    sampled_indices = []
    for label, indices in label_indices.items():
        sampled = np.random.choice(indices, size=min_count, replace=False)
        sampled_indices.extend(sampled)

    sampled_indices = sorted(sampled_indices)
    return dataset.select(sampled_indices)

# Применяем
balanced_train = downsample_dataset(small_train)
print("Новый размер:", len(balanced_train))

balanced_val = downsample_dataset(small_val)
print("Новый размер:", len(balanced_val))

Новый размер: 84
Новый размер: 102


## Smoke test 1: head only

In [11]:
# 1. Замораживаем все слои BERT - обучается только классификатор
for param in model.bert.parameters():
    param.requires_grad = False

print([(param_name) for param_name, tensor in model.named_parameters() if tensor.requires_grad==True])

['classifier.weight', 'classifier.bias']


In [13]:
# Аргументы обучения
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=30,
    per_device_train_batch_size=16,
    learning_rate=5e-5,
    logging_steps=50,
    # no_cuda=False,
)

# Создаём Trainer и передаём в него наш оптимизатор
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
)

# Запускаем обучение
trainer.train()

Step,Training Loss
50,0.913163
100,0.787050
150,0.728507
200,0.728840
250,0.754537
300,0.741973
350,0.738922
400,0.719942
450,0.736527
500,0.723138


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=570, training_loss=0.7549740741127415, metrics={'train_runtime': 159.5199, 'train_samples_per_second': 56.419, 'train_steps_per_second': 3.573, 'total_flos': 1184010379776000.0, 'train_loss': 0.7549740741127415, 'epoch': 30.0})

## Smoke test 2: head + 3 last layers

In [14]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME,
                                                           num_labels=len(dataset["train"].features["labels"].names)
                                                           )

# 2. Размораживаем только последние K слоёв энкодера (например, последние 2)
K = 2
for layer in model.bert.encoder.layer[-K:]:
    for param in layer.parameters():
        param.requires_grad = True

print([(param_name) for param_name, tensor in model.named_parameters() if tensor.requires_grad==True])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you

['bert.embeddings.word_embeddings.weight', 'bert.embeddings.position_embeddings.weight', 'bert.embeddings.token_type_embeddings.weight', 'bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.self.query.weight', 'bert.encoder.layer.0.attention.self.query.bias', 'bert.encoder.layer.0.attention.self.key.weight', 'bert.encoder.layer.0.attention.self.key.bias', 'bert.encoder.layer.0.attention.self.value.weight', 'bert.encoder.layer.0.attention.self.value.bias', 'bert.encoder.layer.0.attention.output.dense.weight', 'bert.encoder.layer.0.attention.output.dense.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.intermediate.dense.weight', 'bert.encoder.layer.0.intermediate.dense.bias', 'bert.encoder.layer.0.output.dense.weight', 'bert.encoder.layer.0.output.dense.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.

In [16]:
# Аргументы обучения
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=30,
    per_device_train_batch_size=16,
    learning_rate=5e-5,
    logging_steps=50,
    # no_cuda=False,
)

# Создаём Trainer и передаём в него наш оптимизатор
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
)

# Запускаем обучение
trainer.train()

Step,Training Loss


KeyboardInterrupt: 

## Smoke test 3: discriminative learning

In [ ]:
from transformers import AdamW

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME,
                                                           num_labels=len(dataset["train"].features["labels"].names)
                                                           )


# создадим кастомный оптимизатор
no_decay = ['bias', 'LayerNorm.weight', 'LayerNorm.bias']
optimizer_grouped_parameters = [
    # Параметры головы классификатора (высокий LR)
    {'params': [p for n, p in model.named_parameters() if 'classifier' in n and not any(nd in n for nd in no_decay)],
     'lr': 5e-4, 'weight_decay': 0.01},
    {'params': [p for n, p in model.named_parameters() if 'classifier' in n and any(nd in n for nd in no_decay)],
     'lr': 5e-4, 'weight_decay': 0.0},

    # Параметры тела BERT (низкий LR)
    {'params': [p for n, p in model.named_parameters() if 'classifier' not in n and not any(nd in n for nd in no_decay)],
     'lr': 5e-5, 'weight_decay': 0.01},
    {'params': [p for n, p in model.named_parameters() if 'classifier' not in n and any(nd in n for nd in no_decay)],
     'lr': 5e-5, 'weight_decay': 0.0},
]

optimizer = AdamW(optimizer_grouped_parameters)


# Аргументы обучения
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=13,
    per_device_train_batch_size=16,
    learning_rate=5e-5,  # Этот параметр будет переопределён нашим оптимизатором
    logging_steps=50,
    no_cuda=False,
)

# Создаём Trainer и передаём в него наш оптимизатор
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    optimizers=(optimizer, None),  # Передаем кастомный оптимизатор
)

# Запускаем обучение
trainer.train()

Some weights of the model checkpoint at DeepPavlov/rubert-base-cased were not used when initializing BertForSequenceClassification: ['cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.decoder.weight', 'cls.predictions.bias', 'cls.seq_relationship.weight', 'cls.predictions.decoder.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were n

{'loss': 0.6829, 'learning_rate': 0.00039878542510121456, 'epoch': 2.63}


                                                 
 40%|████      | 100/247 [09:41<14:09,  5.78s/it]

{'loss': 0.4647, 'learning_rate': 0.0002975708502024291, 'epoch': 5.26}


                                                 
 61%|██████    | 150/247 [14:29<09:28,  5.86s/it]

...

                                                 
 81%|████████  | 200/247 [19:14<04:34,  5.85s/it]

{'loss': 0.0311, 'learning_rate': 9.51417004048583e-05, 'epoch': 10.53}


                                                 
100%|██████████| 247/247 [23:43<00:00,  5.76s/it]

{'train_runtime': 1423.2005, 'train_samples_per_second': 2.74, 'train_steps_per_second': 0.174, 'train_loss': 0.27464680762788063, 'epoch': 13.0}


TrainOutput(global_step=247, training_loss=0.27464680762788063, metrics={'train_runtime': 1423.2005, 'train_samples_per_second': 2.74, 'train_steps_per_second': 0.174, 'train_loss': 0.27464680762788063, 'epoch': 13.0})

## Configure final trainer for whole data

In [12]:
import numpy as np
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    """
    Вычисление weighted F1-score для многоклассовой классификации
    """
    predictions, labels = eval_pred

    # Получаем предсказанные классы
    predicted_classes = np.argmax(predictions, axis=1)

    # Вычисляем weighted F1-score
    f1_weighted = f1_score(labels, predicted_classes, average='weighted')

    return {"f1_weighted": f1_weighted}

In [18]:
from torch.optim import AdamW

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME,
                                                           num_labels=len(dataset["train"].features["labels"].names)
                                                           )

# создадим кастомный оптимизатор
no_decay = ['bias', 'LayerNorm.weight', 'LayerNorm.bias']
optimizer_grouped_parameters = [
    # Параметры головы классификатора (высокий LR)
    {'params': [p for n, p in model.named_parameters() if 'classifier' in n and not any(nd in n for nd in no_decay)],
     'lr': 5e-4, 'weight_decay': 0.01},
    {'params': [p for n, p in model.named_parameters() if 'classifier' in n and any(nd in n for nd in no_decay)],
     'lr': 5e-4, 'weight_decay': 0.0},

    # Параметры тела BERT (низкий LR)
    {'params': [p for n, p in model.named_parameters() if 'classifier' not in n and not any(nd in n for nd in no_decay)],
     'lr': 5e-5, 'weight_decay': 0.01},
    {'params': [p for n, p in model.named_parameters() if 'classifier' not in n and any(nd in n for nd in no_decay)],
     'lr': 5e-5, 'weight_decay': 0.0},
]

optimizer = AdamW(optimizer_grouped_parameters)


# Аргументы обучения
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="steps",
    learning_rate=5e-5,  # Этот параметр будет переопределён нашим оптимизатором
    logging_steps=50,
    # no_cuda=False,
    load_best_model_at_end = True,
)

# Создаём Trainer и передаём в него наш оптимизатор
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=medium_train,
    eval_dataset=tokenized_val,
    optimizers=(optimizer, None),  # Передаем кастомный оптимизатор
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=100)], # если три раза подряд метрика не улучшилась - досрочный останов
)

print("Обучение с валидацией...")
trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you

Обучение с валидацией...


Step,Training Loss,Validation Loss,F1 Weighted
50,0.718547,0.749250,0.637297
100,0.628770,0.631805,0.661607
150,0.561827,0.621510,0.714087
200,0.583549,0.562298,0.748278
250,0.545377,0.470091,0.787322
300,0.549978,0.474833,0.767277
350,0.487333,0.527073,0.786896
400,0.434618,0.499785,0.768642
450,0.429158,0.479463,0.794574
500,0.401216,0.495562,0.793172


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1565, training_loss=0.31511991884761725, metrics={'train_runtime': 6088.9722, 'train_samples_per_second': 8.212, 'train_steps_per_second': 0.257, 'total_flos': 6577835443200000.0, 'train_loss': 0.31511991884761725, 'epoch': 5.0})

In [35]:
import numpy as np
from torch.utils.data import DataLoader

tokenized_val = tokenized_val.with_format('torch')
tokenized_val = tokenized_val.select_columns(['input_ids', 'attention_mask', 'labels'])
test_loader = DataLoader(tokenized_val, batch_size=16)
model.eval()  # Переводим модель в режим оценки

all_predictions = []
all_labels = []

with torch.no_grad():  # Отключаем расчет градиентов
    for batch in test_loader:
        # Переносим данные на устройство (GPU/CPU)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Получаем логиты (выходы модели)
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        # Превращаем логиты в предсказанные классы
        predictions = torch.argmax(logits, dim=-1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Преобразуем списки в массивы NumPy для удобства
all_predictions = np.array(all_predictions)
all_labels = np.array(all_labels)

In [36]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

# Рассчитываем метрики
accuracy = accuracy_score(all_labels, all_predictions)
f1 = f1_score(all_labels, all_predictions, average='weighted')  # Для многоклассовой классификации
precision = precision_score(all_labels, all_predictions, average='weighted')
recall = recall_score(all_labels, all_predictions, average='weighted')

# Выводим результаты
print(f'Точность (Accuracy): {accuracy:.4f}')
print(f'F1-мера (Weighted): {f1:.4f}')
print(f'Точность (Precision): {precision:.4f}')
print(f'Полнота (Recall): {recall:.4f}')

# Детальный отчет по каждому классу
print("\nClassification Report:")
print(classification_report(all_labels, all_predictions))

Точность (Accuracy): 0.7901
F1-мера (Weighted): 0.7964
Точность (Precision): 0.8123
Полнота (Recall): 0.7901

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.51      0.62       950
           1       0.90      0.90      0.90      5459
           2       0.30      0.42      0.35       910

    accuracy                           0.79      7319
   macro avg       0.66      0.61      0.62      7319
weighted avg       0.81      0.79      0.80      7319

